# StorageGuard: scale study (larger models)

Run this after the main study (Qwen2.5-7B and Ministral 3 8B). It asks: **does a larger model still need the guard?** Same family (Qwen2.5), same tool format, increasing size, the five main configurations on the 104 test tasks.

| Model | Weights | Why this precision |
|---|---|---|
| Qwen2.5-7B (done) | BF16 | main study |
| **Qwen2.5-7B AWQ** | 4-bit, 5.6 GB | control: what 4-bit weights alone change |
| **Qwen2.5-14B** | BF16, 29.5 GB | full precision |
| **Qwen2.5-32B GPTQ-Int8** | 8-bit, 35.1 GB | official, near-lossless; BF16 (65.5 GB) leaves no KV cache |
| **Qwen2.5-72B AWQ** | 4-bit, 41.6 GB | official; 8-bit (77.1 GB) does not fit |

On a 40 GB A100, cell 2 switches to the 40 GB ladder (14B 8-bit, 32B 4-bit) by itself.

| Cells | What | Time on an A100 80 GB (estimate) |
|---|---|---|
| 1-3 | GPU check, Drive, install (a new runtime starts empty) | ~10 min |
| 4 | per model: download, smoke test, main runs (1 run each) | ~1.5-2 h (~3.5 h with `SCALE_RUNS = 3`) |
| 5-6 | report over all models so far, download | 2 min |
| 7-9 | optional: Llama 3.1, if Meta has approved it by now | ~30 min |

**Before you start**

1. If another notebook is still connected, stop its runtime there (**Runtime -> Disconnect and delete runtime**), so two A100s are not billed at once.
2. Upload the newest `storageguard_colab.zip` into `MyDrive/storageguard/` (**Replace existing file**). Your earlier results are in `MyDrive/storageguard/state/` and are restored automatically; nothing finished is run again.
3. **Runtime -> Change runtime type -> A100 GPU**. The `HF_TOKEN` secret is only needed for the Llama section.

## 1. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU")
assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU"
if "A100" not in gpu and "H100" not in gpu:
    print("WARNING: not an A100/H100. 7-8B models still fit on a 24 GB GPU (L4), but expect it to be slower.")

## 2. Settings, Google Drive and the project

In [ ]:
# ---- settings -------------------------------------------------------------
BACKEND    = "vllm"
SCALE      = "auto"      # "auto" = the ladder for this GPU; or a list, e.g. "qwen2.5-14b,qwen2.5-32b-int8"
SCALE_RUNS = 1           # runs per configuration; 3 matches the main models (about twice as long overall)
LLAMA      = "llama3.1-8b"                          # optional last section, once Meta has approved it
DRIVE_DIR  = "/content/drive/MyDrive/storageguard"  # where storageguard_colab.zip is
# ---------------------------------------------------------------------------
LADDER_80GB = "qwen2.5-7b-awq,qwen2.5-14b,qwen2.5-32b-int8,qwen2.5-72b-awq"
LADDER_40GB = "qwen2.5-7b-awq,qwen2.5-14b-int8,qwen2.5-32b-awq"
if SCALE == "auto":
    import subprocess
    try:
        _mib = float(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                                    capture_output=True, text=True).stdout.split()[0])
    except Exception:
        _mib = 0.0
    SCALE = LADDER_80GB if _mib >= 70000 else LADDER_40GB
    print(f"GPU memory {_mib / 1024:.0f} GB -> scale ladder: {SCALE}")

import os, sys, time, zipfile, pathlib, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
bundles = sorted(pathlib.Path(DRIVE_DIR).glob("storageguard_colab*.zip"), key=os.path.getmtime)
assert bundles, f"Upload storageguard_colab.zip to {DRIVE_DIR} first"
bundle = bundles[-1]                          # the newest upload, also if Drive kept an older copy
zipfile.ZipFile(bundle).extractall("/content")  # always the newest code; results are never in the zip
print("code from", bundle.name, "uploaded", time.strftime("%Y-%m-%d %H:%M", time.localtime(bundle.stat().st_mtime)))
os.chdir("/content/storageguard")
os.environ["SG_DRIVE_STATE"] = f"{DRIVE_DIR}/state"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as exc:  # secret missing or notebook access not granted
    print(f"No HF_TOKEN ({type(exc).__name__}). Qwen and Mistral will work; Llama downloads will fail.")

def run(*args):
    """Run the driver, stream its output here, and stop the notebook if it fails."""
    p = subprocess.Popen([sys.executable, "-u", "colab/sg_colab.py", *map(str, args)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed with exit code {p.returncode}; read the messages above")

def show_report():
    from IPython.display import Markdown, Image, display
    display(Markdown(pathlib.Path("results/REPORT.md").read_text()))
    for png in sorted(pathlib.Path("paper/figures").glob("*.png")):
        print(png.name)
        display(Image(filename=str(png), width=900))

def download_results():
    run("pack")   # zips results/, paper/, data/bench and configs into DRIVE_DIR
    from google.colab import files
    latest = max(pathlib.Path(DRIVE_DIR).glob("storageguard_results_*.zip"), key=os.path.getmtime)
    print("saved to Drive:", latest)
    files.download(str(latest))

print("project ready at", os.getcwd())

## 3. Install (about 5-10 minutes)

Creates two isolated environments: one for the experiment harness (Python 3.11) and one for vLLM (Python 3.12), then runs the fast unit tests.

In [ ]:
run("setup", "--backend", BACKEND)

## 4. Scale study: smoke test and main runs, model by model

Smallest model first. For each model: check GPU memory and download access (seconds), download (when the disk runs short, cached weights of finished models are deleted), smoke test, then the main runs only if the smoke test passed. A model that fails is reported and skipped; the others continue. At the end the report is rebuilt over all models.

Keep the tab open. If the session disconnects: run cells 1, 2, 3, then this cell again; finished episodes are skipped.

In [ ]:
run("all", "--backend", BACKEND, "--models", SCALE, "--main-runs", SCALE_RUNS, "--smoke-first")

## 5. Report (all models so far)

In [ ]:
show_report()

## 6. Download the results

A copy is also saved in `MyDrive/storageguard/`.

In [ ]:
download_results()

---
## Optional: Llama 3.1 (only once Meta has approved it)

The model page (<https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct>) must say *You have been granted access*, and the `HF_TOKEN` secret must have **Notebook access** (if you add it now, run cell 2 again).

## 7. Has Meta approved it yet? (seconds)

In [ ]:
run("access", "--models", LLAMA)

## 8. Llama: smoke test and main runs (3 runs, like the other main models; ~30 min)

In [ ]:
run("all", "--backend", BACKEND, "--models", LLAMA, "--main-runs", 3, "--smoke-first")

## 9. Final report and download

In [ ]:
show_report()
download_results()

## Troubleshooting

| Symptom | Fix |
|---|---|
| Cell 4 stops at once: `needs a ... GB+ GPU` | This runtime has a smaller GPU (e.g. a 40 GB A100): set `SCALE = "auto"` (or the 40 GB ladder) in cell 2 and run cells 2 and 4 again. |
| A model: `SMOKE FAIL` | Its runs are skipped and the others continue. The output shows what the model wrote; report it as a limitation. |
| `CUDA out of memory` for the 72B model | Run cell 4 again with `"--concurrency", 4` added: finished models are skipped. |
| Session disconnected | Run cells 1, 2, 3, then the cell that was running. Nothing finished is lost. |
| Download fails: no space left on device | Run the cell again: finished models' cached weights are removed first. |
| Want to check progress | `!find results/runs/main -name '*.json' | wc -l` in a new cell. |